# Fig 2A — Bootstrap Correlation Computation

Computes per-cancer-type bootstrapped Pearson correlations between each axis score and log₁₀(TMB), applies FDR correction, and saves results to `results/tables/score_corr_per_type_summary.csv`.

**Run this notebook to regenerate the CSV; the figure notebook reads from it.**

In [ ]:
import os
import socket

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from statsmodels.stats.multitest import fdrcorrection

In [ ]:
hostname = socket.gethostname()
if "scg" in hostname:
    BASE_DIR = "/labs/mwinslow/Karen/protein-master/v2_pancan_batch_corrected"
else:
    BASE_DIR = os.path.expanduser("~/Documents/research/scg/protein-master/v2_pancan_batch_corrected")

DATA_DIR  = f"{BASE_DIR}/analysis/01_axis_analysis/bh/axis_sample_scores"
OUT_TABLE = f"{BASE_DIR}/results/tables/score_corr_per_type_summary.csv"
print(f"Will save to: {OUT_TABLE}")

In [ ]:
df_axis = pd.read_parquet(f"{DATA_DIR}/tcga_xena/axis_sample_level.parquet")
df_axis.shape

## Config

In [ ]:
SCORES = ["TMB_axis_score", "pqc_axis_score", "non_pqc_axis_score"]

SCORE_LABELS = {
    "TMB_axis_score":     "TMB axis",
    "pqc_axis_score":     "PQC axis",
    "non_pqc_axis_score": "Non-PQC axis",
}

N_BOOT   = 1000
SEED     = 2026
MIN_N    = 10    # minimum samples per cancer type to include
FDR_THRESH = 0.05

## Bootstrap function

In [ ]:
def bootstrap_corr_per_type(df, scores=SCORES, n_boot=N_BOOT, seed=SEED, min_n=MIN_N):
    """Bootstrap Pearson r between each score and log_mut, per cancer type."""
    rng = np.random.default_rng(seed)
    results = []

    for ct, df_ct in df.groupby("cancer_type"):
        if len(df_ct) < min_n:
            continue
        n = len(df_ct)

        for score_col in scores:
            obs = df_ct[["log_mut", score_col]].corr().iloc[0, 1]

            boot_corrs = np.array([
                df_ct.sample(n=n, replace=True, random_state=rng.integers(1_000_000_000))
                     [["log_mut", score_col]].corr().iloc[0, 1]
                for _ in range(n_boot)
            ])

            ci_low  = np.percentile(boot_corrs, 2.5)
            ci_high = np.percentile(boot_corrs, 97.5)

            # sign-based two-sided p-value
            p_left  = (np.sum(boot_corrs <= 0) + 1) / (n_boot + 1)
            p_right = (np.sum(boot_corrs >= 0) + 1) / (n_boot + 1)
            p_val   = min(2 * min(p_left, p_right), 1.0)

            results.append({
                "cancer_type": ct,
                "score":       score_col,
                "r":           obs,
                "ci_low":      ci_low,
                "ci_high":     ci_high,
                "p_value":     p_val,
                "n":           n,
            })

    return pd.DataFrame(results)

## Run bootstrap

This takes a few minutes (1000 resamples × 32 cancer types × 3 scores).

In [ ]:
bs_corr_df = bootstrap_corr_per_type(df_axis)
print(f"{len(bs_corr_df)} rows")
bs_corr_df.head()

In [ ]:
bs_corr_df["fdr"] = (
    bs_corr_df
    .groupby("score")["p_value"]
    .transform(lambda p: fdrcorrection(p)[1])
)

bs_corr_df["score_label"] = bs_corr_df["score"].map(SCORE_LABELS)
bs_corr_df.head()

In [ ]:
bs_corr_df.to_csv(OUT_TABLE, index=False)
print(f"Saved: {OUT_TABLE}")